# QuestionGen UI Launcher

Primary staff-facing Colab launcher. This notebook keeps `main` as the stable default target, uses a notebook-side allowlist for pushed branches, and loads repo code from `src/` instead of reinstalling `questiongen` on routine reruns.

After a clean runtime, set `BOOTSTRAP_ENV=True` once if the third-party packages are not installed yet. Normal rerun path: leave `BOOTSTRAP_ENV=False` and `RESET_REPO=False`. For pushed branch validation, temporarily add the branch to `REPO_BRANCH_OPTIONS`, set `REPO_BRANCH`, and use `RESET_REPO=True`. If `questiongen` was already imported in this kernel, restart the runtime before launching the refreshed app.


## 1. Mount Drive And Define Standard Paths


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Edit DATA_DIR only if your runtime folder lives somewhere else in Drive.
DATA_DIR = Path("/content/drive/MyDrive/Work/Ebenezer Related/QuestionGenData")
SECRETS_DIR = DATA_DIR / "secrets"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
LOGS_DIR = DATA_DIR / "logs"

for folder in [SECRETS_DIR, INPUT_DIR, OUTPUT_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("SECRETS_DIR:", SECRETS_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Minimal Settings


In [ ]:
API_KEY_FILE = SECRETS_DIR / "api_key.txt"
DEFAULT_DRIVE_CSV = INPUT_DIR / "questions.csv"
GRADIO_OUTPUT_DIR = OUTPUT_DIR / "gradio"

print("API_KEY_FILE:", API_KEY_FILE)
print("Default Drive CSV shown in UI:", DEFAULT_DRIVE_CSV)
print("Default Gradio output dir shown in UI:", GRADIO_OUTPUT_DIR)


## 3. Load Secrets From Drive


In [ ]:
import os
from pathlib import Path

def load_api_keys(filepath: str | Path) -> None:
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(
            f"API key file not found: {filepath}\n"
            "Create it with lines like OPENAI_API_KEY=sk-..."
        )

    with filepath.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

load_api_keys(API_KEY_FILE)

MODEL_NAME = os.getenv("QUESTIONGEN_MODEL", "gpt-5-mini")
TEMPERATURE = float(os.getenv("QUESTIONGEN_TEMPERATURE", "0"))

print("Loaded env vars:")
print("- OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "missing")
print("- QUESTIONGEN_MODEL:", MODEL_NAME)
print("- QUESTIONGEN_TEMPERATURE:", TEMPERATURE)


## 4. Advanced Settings

`main` is the stable default branch. Add another branch to `REPO_BRANCH_OPTIONS` only when that branch is already pushed and you intentionally want Colab to validate it.

`BOOTSTRAP_ENV` is the one-time or infrequent third-party dependency path. `RESET_REPO` deletes the existing clone and reclones the selected pushed branch. Routine reruns should usually keep both set to `False`.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/AwkAwkAardvark/QuestionGen.git"
REPO_BRANCH_OPTIONS = [
    "main",
]
REPO_BRANCH = REPO_BRANCH_OPTIONS[0]
BOOTSTRAP_ENV = False
RESET_REPO = False
REPO_DIR = Path("/content/QuestionGen")

if REPO_BRANCH not in REPO_BRANCH_OPTIONS:
    raise ValueError(
        f"REPO_BRANCH must be one of {REPO_BRANCH_OPTIONS}. "
        "Update REPO_BRANCH_OPTIONS only when you intentionally want Colab to target another pushed branch."
    )

print("REPO_URL:", REPO_URL)
print("REPO_BRANCH_OPTIONS:", REPO_BRANCH_OPTIONS)
print("REPO_BRANCH:", REPO_BRANCH)
print("BOOTSTRAP_ENV:", BOOTSTRAP_ENV)
print("RESET_REPO:", RESET_REPO)
print("REPO_DIR:", REPO_DIR)


## 5. Bootstrap, Refresh, And Launch Gradio

This notebook does not run `run_batch_files(...)` directly before launch. The Gradio app is the active UI surface, and repo refresh is intentionally separated from in-kernel imports so pushed branch validation stays predictable.


In [ ]:
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys

BOOTSTRAP_PACKAGES = [
    "gradio",
    "langchain-core",
    "langchain-openai",
    "langgraph",
    "pandas",
    "pydantic",
]
SRC_DIR = REPO_DIR / "src"

def ensure_clean_questiongen_import(action_name: str) -> None:
    if "questiongen" in sys.modules and (RESET_REPO or BOOTSTRAP_ENV):
        raise RuntimeError(
            "questiongen is already imported in this notebook kernel. The requested bootstrap/repo refresh step completed and the updated pushed branch source is ready, but "
            f"{action_name} requires a runtime restart for a clean import. Fresh-subprocess tests can still validate the updated pushed branch."
        )

if BOOTSTRAP_ENV:
    get_ipython().run_line_magic("pip", "install -q " + " ".join(BOOTSTRAP_PACKAGES))
    print("Bootstrapped third-party dependencies only.")
else:
    print("Skipped third-party dependency bootstrap.")

if RESET_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    print("Deleted existing repo clone for a clean refresh.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print("Cloned branch:", REPO_BRANCH)
else:
    print("Reusing existing repo clone:", REPO_DIR)

importlib.invalidate_caches()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if importlib.util.find_spec("questiongen") is None:
    raise ModuleNotFoundError(
        "questiongen is not importable from REPO_DIR / 'src'. Check the bootstrap/repo setup output before continuing."
    )

os.environ["QUESTIONGEN_DATA_DIR"] = str(DATA_DIR)
os.environ["QUESTIONGEN_API_KEY_PATH"] = str(API_KEY_FILE)
os.environ["QUESTIONGEN_DRIVE_INPUT_CSV"] = str(DEFAULT_DRIVE_CSV)
os.environ["QUESTIONGEN_OUTPUT_DIR"] = str(GRADIO_OUTPUT_DIR)

print("Prepared branch source:", REPO_BRANCH)
print("Repo dir:", REPO_DIR)
print("Source path added:", SRC_DIR)
print("Python executable:", sys.executable)

ensure_clean_questiongen_import("in-kernel Gradio launch")

from questiongen.config import ensure_runtime_dependencies
from questiongen.ui.gradio_app import create_app

ensure_runtime_dependencies(
    bootstrap_hint="In Colab, rerun this setup cell once with BOOTSTRAP_ENV=True, then restart the runtime if questiongen was already imported."
)

app = create_app()
app.launch(debug=True)
